# Smart MCQ Solver Challenge
**Roll No:** 23f3004491 | **Term:** T2-2026 | **Metric:** MAP@3 | **Cutoff:** 0.73

Each question has a prompt + 5 options (A-E); predict the top-3 answers in ranked order.

### Key data insights (drive all model choices)
1. Distractor options are **minimal edits** of each other (~34% of questions have options
   >70% character-identical) - similarity ranking fails; options must be compared side-by-side.
2. Questions test **factual science knowledge** - small encoders lack it; a 7B LLM contains it.
3. **Train contains only 415 unique questions in 2000 rows** (same question re-wrapped in
   one of exactly 5 start-phrases x 4 end-phrases), and **~94% of test rows duplicate a train
   question** once every wrapper variant is stripped correctly (Sec. 5) - enabling a lookup
   strategy, and requiring a dedup-aware validation split to avoid leakage.

### Model progression (validation MAP@3)
| # | Model | Type | MAP@3 |
|---|---|---|---|
| 1 | TF-IDF + cosine | from scratch | 0.31 |
| 2 | MiniLM bi-encoder | pretrained | 0.40 |
| 3 | NLI cross-encoder | pretrained, zero-shot | 0.56 |
| 4 | LoRA DeBERTa-v3-large (multiple-choice) | fine-tuned | 0.59 |
| 5 | Qwen2.5-7B-Instruct | large LLM, zero-shot | 0.90* |
| 6 | **Hybrid: train-lookup + Qwen** | final submission | **leaderboard** |

*inflated by duplicate leakage in the naive split; see Section 5 for the honest split.

## 1. Environment Setup

In [1]:
!pip install -q -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" "sentence-transformers==3.3.1"
print("Packages installed (no restart needed). Continue running all cells.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 whic

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import warnings
warnings.filterwarnings('ignore')

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

OPTIONS = ['A', 'B', 'C', 'D', 'E']
SEED = 42

GPU available: True
GPU name: Tesla T4


## 1b. Experiment Configuration

Every experiment in this notebook is a flag here. Flip a flag, `Run All`, read the
proxy-evaluation section, and only spend a Kaggle submission once the proxy agrees.

- `LLM_NAME` / `RUN_14B`: 7B baseline, or shard 14B across the two T4s.
- `TTA_ORDERS`: score each question under N option orderings and average (position-bias TTA).
- `EXTRACTION`: `space` (` A`), `plain` (`A`), or `both` (average of the two) letter readouts.
- `GATING`: how retrieval and the LLM are combined (`off` / `always` / `agreement`).
  **The default `always` reproduces the 0.73607 leaderboard baseline** - exact-text
  lookups are a strong signal and beat overriding them with the 7B's guess.
- `LOOKUP_VOTE`: `last` (baseline dict, last copy wins) or `majority` (vote across copies).
- `ENSEMBLE_7B_14B`: average 7B and 14B probabilities for the submission.


In [3]:
import random

CONFIG = {
    "LLM_NAME":         "Qwen/Qwen2.5-7B-Instruct",
    "RUN_14B":          False,
    "LLM_NAME_14B":     "Qwen/Qwen2.5-14B-Instruct",
    "TTA_ORDERS":       1,
    "LLM_VALID_SAMPLE": 200,
    "EXTRACTION":       "space",
    "GATING":           "always",
    "LOOKUP_VOTE":      "last",
    "ENSEMBLE_7B_14B":  False,
    "RUN_MODEL2":       True,
    "MODEL2_NAME":      "mistralai/Mistral-7B-Instruct-v0.3",
    "ENSEMBLE_W_QWEN_M2": (0.5, 0.5),
    "ENSEMBLE_WEIGHTS": (0.5, 0.5),
    "PROXY_EVAL":       False,
    "PSEUDO_TEST_EVAL": False,
    "SEED":             SEED,
}


def seed_everything(seed=SEED):
    """Seed python, numpy and torch so every run is reproducible."""
    random.seed(seed)
    import numpy as _np
    _np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        from transformers import set_seed
        set_seed(seed)
    except Exception:
        pass


seed_everything(CONFIG["SEED"])
print("Config ready:", {k: CONFIG[k] for k in ("LLM_NAME", "RUN_14B", "TTA_ORDERS", "EXTRACTION", "GATING", "ENSEMBLE_7B_14B")})

2026-07-29 11:48:45.340192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785325725.531283      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785325725.587361      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785325726.012223      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785325726.012265      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785325726.012268      23 computation_placer.cc:177] computation placer alr

Config ready: {'LLM_NAME': 'Qwen/Qwen2.5-7B-Instruct', 'RUN_14B': False, 'TTA_ORDERS': 1, 'EXTRACTION': 'space', 'GATING': 'always', 'ENSEMBLE_7B_14B': False}


## 2. Load Data

In [4]:
import pandas as pd

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train  = pd.read_csv(f"{BASE}/train.csv")
test   = pd.read_csv(f"{BASE}/test.csv")
sample = pd.read_csv(f"{BASE}/sample_submission.csv")

print("TRAIN:", train.shape, "| TEST:", test.shape)

TRAIN: (2000, 8) | TEST: (500, 7)


## 3. Quick EDA — the two facts that matter

1. **Option near-duplication** — distractors are minimal edits, so the discriminating
   signal is one or two tokens. This kills similarity approaches.
2. **Answer distribution** — mildly skewed to B/C; sets the naive floor (~0.30 MAP@3).

In [5]:
import numpy as np
import difflib
from itertools import combinations

sims = []
for i in range(300):
    r = train.iloc[i]
    pair_sims = [difflib.SequenceMatcher(None, str(r[a]), str(r[b])).ratio()
                 for a, b in combinations(OPTIONS, 2)]
    sims.append(np.mean(pair_sims))
sims = np.array(sims)
print(f"Mean pairwise option similarity: {sims.mean():.3f}")
print(f"Questions with near-identical options (>0.7): {(sims > 0.7).mean()*100:.0f}%")

print("\nAnswer distribution:")
print(train['answer'].value_counts().sort_index().to_string())

total_len = train['prompt'].str.split().str.len() + train[OPTIONS].apply(
    lambda col: col.str.split().str.len()).max(axis=1)
print(f"\nPrompt+longest-option words: median={total_len.median():.0f}, "
      f"95th pct={total_len.quantile(0.95):.0f}, max={total_len.max():.0f}")

Mean pairwise option similarity: 0.588
Questions with near-identical options (>0.7): 34%

Answer distribution:
answer
A    369
B    490
C    459
D    358
E    324

Prompt+longest-option words: median=45, 95th pct=84, max=148


## 4. MAP@3 Evaluation Metric

Correct answer at rank 1 → 1.0, rank 2 → 0.5, rank 3 → 0.333, else 0.

In [6]:
def average_precision_at_3(true_label, predicted_labels):
    """Score one question: 1/(rank+1) if the correct answer is in the top-3, else 0."""
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

def mean_average_precision_at_3(true_labels, predicted_lists):
    """Average AP@3 across all questions."""
    return np.mean([average_precision_at_3(t, p)
                    for t, p in zip(true_labels, predicted_lists)])

assert average_precision_at_3('B', ['B','A','C']) == 1.0
assert average_precision_at_3('B', ['A','B','C']) == 0.5
assert abs(average_precision_at_3('B', ['A','C','B']) - 1/3) < 1e-9
assert average_precision_at_3('B', ['A','C','D']) == 0.0
print("MAP@3 scorer verified.")

MAP@3 scorer verified.


## 5. Prompt Normalization & Leakage-Free Split

The same core question appears multiple times in train with different wrapper phrases
("Pick the best possible answer:" vs "Select the most accurate option:"). A naive random
split puts copies of the same question on both sides, inflating validation (we measured
0.90 vs a 0.69 leaderboard). Fix: strip wrappers to get the **core question**, then split
by unique core so no question appears in both train and validation.

**Correction:** the wrapper lists below were originally guessed and got 2 of 5 start-phrases
and 2 of 4 end-phrases wrong (e.g. listed "choose the right answer:" when the data actually
uses "choose the correct answer:"; never listed "from the following choices." or "based on
the given context." at all). That silently left ~55% of train rows only partially stripped,
undercounting duplicates and starving the retrieval lookup in Section 12. The lists are now
verified directly against the raw prompt text (every start-phrase before the first `:` and
every end-phrase after the last `?`, counted across train+test) before being hardcoded.

In [7]:
import collections

_prefixes, _suffixes = collections.Counter(), collections.Counter()
for _p in pd.concat([train['prompt'], test['prompt']]):
    _low = str(_p).strip().lower()
    _colon = _low.find(':')
    if 0 < _colon < 60:
        _prefixes[_low[:_colon + 1]] += 1
    _q = _low.rfind('?')
    if _q != -1 and _q < len(_low) - 1:
        _tail = _low[_q + 1:].strip()
        if _tail:
            _suffixes[_tail] += 1

print("Start-wrapper phrases actually present (text before the first ':'):")
for k, v in _prefixes.most_common():
    print(f"  {v:5d}  {k!r}")
print("\nEnd-wrapper phrases actually present (text after the last '?'):")
for k, v in _suffixes.most_common():
    print(f"  {v:5d}  {k!r}")

Start-wrapper phrases actually present (text before the first ':'):
    360  'pick the best possible answer:'
    359  'select the most accurate option:'
    355  'determine the correct option:'
    349  'identify the correct statement:'
    348  'choose the correct answer:'

End-wrapper phrases actually present (text after the last '?'):
    444  'among the listed options.'
    434  'based on the given context.'
    421  'from the following choices.'
    391  'carefully.'


In [8]:
import re
from sklearn.model_selection import train_test_split

# Verified against the raw prompt text (see EDA below): exactly 5 start-wrappers and
# 4 end-wrappers are actually used. The previous lists guessed wrong variants (e.g.
# "choose the right answer:" instead of "choose the correct answer:", and were missing
# "from the following choices." / "based on the given context." entirely) - which
# silently fragmented ~55% of train cores and starved the retrieval lookup below.
WRAPPERS_START = ["pick the best possible answer:", "select the most accurate option:",
    "identify the correct statement:", "determine the correct option:", "choose the correct answer:"]
WRAPPERS_END = ["among the listed options.", "among the listed options",
    "from the following choices.", "from the following choices",
    "based on the given context.", "based on the given context",
    "carefully.", "carefully"]

def normalize_prompt(p):
    """Strip wrapper phrases to recover the core question text."""
    p = str(p).strip().lower()
    changed = True
    while changed:
        changed = False
        for w in WRAPPERS_START:
            if p.startswith(w):
                p = p[len(w):].strip(); changed = True
        for w in WRAPPERS_END:
            if p.endswith(w):
                p = p[:-len(w)].strip(); changed = True
    return re.sub(r'\s+', ' ', p)

train['core'] = train['prompt'].apply(normalize_prompt)
test['core'] = test['prompt'].apply(normalize_prompt)

unique_cores = train['core'].unique()
core_train, core_valid = train_test_split(unique_cores, test_size=0.2, random_state=SEED)

train_df = train[train['core'].isin(core_train)].reset_index(drop=True)
valid_df = train[train['core'].isin(core_valid)].drop_duplicates('core').reset_index(drop=True)

print("Unique core questions:", len(unique_cores), "of", len(train), "rows")
print("Train rows:", len(train_df), "| Valid rows (deduped):", len(valid_df))
print("Core overlap between splits:", len(set(train_df['core']) & set(valid_df['core'])))

Unique core questions: 415 of 2000 rows
Train rows: 1590 | Valid rows (deduped): 83
Core overlap between splits: 0


In [9]:
novel_mask = ~train['core'].isin(set(test['core']))
proxy_df = (train[novel_mask]
            .drop_duplicates('core')
            .reset_index(drop=True))

novel_test_rows = (~test['core'].isin(train['core'])).sum()
print("Novel-proxy questions (train cores absent from test):", len(proxy_df))
print(f"Test rows with NO matching core in train at all: {novel_test_rows} ({100*novel_test_rows/len(test):.0f}%)")
print("These proxy questions carry ground-truth labels and share no core with test, so they are"
      " the closest labelled stand-in for that genuinely novel slice of test.")

Novel-proxy questions (train cores absent from test): 156
Test rows with NO matching core in train at all: 10 (2%)
These proxy questions carry ground-truth labels and share no core with test, so they are the closest labelled stand-in for that genuinely novel slice of test.


## 6. W&B Setup

Every model below logs a run so all runs are comparable on the same metrics (MAP@3, accuracy, F1). API key lives in a Kaggle Secret — safe to commit this notebook.

In [10]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "23f3004491-t22026"
wandb.login(key=os.environ["WANDB_API_KEY"])

from sklearn.metrics import accuracy_score, f1_score

def log_model_run(name, model_type, valid_df, predictions, config=None):
    """Log one model's validation performance to W&B as its own run.

    Logs MAP@3 plus top-1 accuracy and macro-F1 so all runs share common
    metrics for comparison (a project requirement).
    """
    truth = valid_df['answer'].tolist()
    top1  = [p[0] for p in predictions]

    metrics = {
        "map@3":      mean_average_precision_at_3(truth, predictions),
        "accuracy":   accuracy_score(truth, top1),
        "f1_macro":   f1_score(truth, top1, average='macro'),
    }

    run = wandb.init(project=os.environ["WANDB_PROJECT"], name=name,
                     config={"model_type": model_type, **(config or {})},
                     reinit=True)
    wandb.log(metrics)
    run.finish()

    print(f"[{name}] " + " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    return metrics["map@3"]

print("W&B ready.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3004491 (23f3004491-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready.


## 7. Model 1 — TF-IDF + Cosine Similarity *(from scratch)*

Word-frequency vectors, no semantics. Expected weak: when options differ by one word,
their TF-IDF vectors are nearly identical.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_predict(df):
    """Rank options by TF-IDF cosine similarity to the prompt."""
    predictions = []
    for _, row in df.iterrows():
        texts = [row['prompt']] + [row[o] for o in OPTIONS]
        vectors = TfidfVectorizer(stop_words='english').fit_transform(texts)
        scores = cosine_similarity(vectors[0], vectors[1:])[0]
        predictions.append([OPTIONS[i] for i in np.argsort(scores)[::-1]])
    return predictions

tfidf_preds = tfidf_predict(valid_df)
tfidf_score = log_model_run("tfidf-cosine", "from-scratch", valid_df, tfidf_preds)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run 46ajsy7a
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_114902-46ajsy7a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tfidf-cosine
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/46ajsy7a
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.21687
wandb: f1_macro 0.2138

[tfidf-cosine] map@3=0.3574 | accuracy=0.2169 | f1_macro=0.2138


## 8. Model 2 — MiniLM Bi-Encoder *(pretrained)*

Semantic embeddings, prompt and options embedded **separately**. Also expected to underperform
here: near-identical option texts → near-identical embeddings → no ranking signal.

In [12]:
from sentence_transformers import SentenceTransformer, util

bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_predict(df, model):
    """Rank options by semantic cosine similarity to the prompt."""
    prompt_embs = model.encode(df['prompt'].tolist(), convert_to_tensor=True)
    option_embs = {o: model.encode(df[o].tolist(), convert_to_tensor=True) for o in OPTIONS}

    predictions = []
    for i in range(len(df)):
        scores = [util.cos_sim(prompt_embs[i], option_embs[o][i]).item() for o in OPTIONS]
        predictions.append([OPTIONS[j] for j in np.argsort(scores)[::-1]])
    return predictions

bi_preds = embedding_predict(valid_df, bi_encoder)
bi_score = log_model_run("minilm-bi-encoder", "pretrained-zero-shot", valid_df, bi_preds,
                         config={"base_model": "all-MiniLM-L6-v2"})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

wandb: setting up run ipqil09k
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_114937-ipqil09k
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run minilm-bi-encoder
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/ipqil09k
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.3012
wandb: f1_macro 0.29351
wandb:    map@3 0.44378
wandb: 
wandb: 🚀 View run minilm-bi-encoder at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/ipqil09k
wandb: ⭐️ View project at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: Sync

[minilm-bi-encoder] map@3=0.4438 | accuracy=0.3012 | f1_macro=0.2935


## 9. Model 3 — NLI Cross-Encoder *(pretrained, zero-shot)*

Reads (prompt, option) **together** via cross-attention and scores entailment.
First model that can attend to the one-word differences between options.

In [13]:
from sentence_transformers import CrossEncoder

ce_model = CrossEncoder('cross-encoder/nli-deberta-v3-small')

def cross_encoder_predict(df, model):
    """Rank options by NLI entailment score against the prompt."""
    all_pairs = [(row['prompt'], row[o]) for _, row in df.iterrows() for o in OPTIONS]
    logits = model.predict(all_pairs, batch_size=64)
    entail = logits[:, 1].reshape(len(df), 5)

    return [[OPTIONS[j] for j in np.argsort(scores)[::-1]] for scores in entail]

ce_preds = cross_encoder_predict(valid_df, ce_model)
ce_score = log_model_run("nli-cross-encoder", "pretrained-zero-shot", valid_df, ce_preds,
                         config={"base_model": "cross-encoder/nli-deberta-v3-small"})

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_115008-fpgd6ljd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run nli-cross-encoder
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/fpgd6ljd
wandb: updating run metadata; uploading summary
wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.42169
wandb: f1_macro 0.40198
wandb:    map@3 0.54016
wandb: 
wandb: 🚀 View run nli-cross-encoder at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/fpgd6ljd
wandb: ⭐️ View 

[nli-cross-encoder] map@3=0.5402 | accuracy=0.4217 | f1_macro=0.4020


## 10. Model 4 — Multiple-Choice LoRA Fine-Tune *(model of choice)*

### Why this architecture
Previous fine-tuning scored each (prompt, option) pair **independently** — but with
minimal-edit distractors, correctness is only defined *relative to the alternatives*.
`AutoModelForMultipleChoice` encodes all 5 (prompt, option) pairs and applies a
**softmax across the 5 options**, forcing a direct comparison. Side benefits:
- No 80/20 class imbalance (the label is just the correct option index 0–4).
- Directly optimizes ranking — which is what MAP@3 measures.

### Why DeBERTa-v3-large
The questions test factual science knowledge; the 140M-param small model plateaued
because it lacks that knowledge. The 400M large variant is the standard backbone for
this task family. Fitting it on a T4 requires small batches + gradient accumulation
(= Milestone 4's "managing GPU memory and batch sizes").

### 10.1 Tokenize in multiple-choice format
Each example becomes 5 parallel (prompt, option) encodings, label = index of the answer.

In [14]:
from transformers import AutoTokenizer
from datasets import Dataset

MC_MODEL = "microsoft/deberta-v3-large"
MAX_LEN  = 192

tokenizer = AutoTokenizer.from_pretrained(MC_MODEL)

def preprocess_mc(examples):
    """Tokenize one batch into multiple-choice format:
    input_ids shape (batch, 5, seq_len), label = correct option index."""
    first  = [[p] * 5 for p in examples['prompt']]
    second = [[examples[o][i] for o in OPTIONS]
              for i in range(len(examples['prompt']))]

    flat_first  = sum(first, [])
    flat_second = sum(second, [])
    tok = tokenizer(flat_first, flat_second,
                    truncation=True, max_length=MAX_LEN, padding='max_length')

    grouped = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tok.items()}
    if 'answer' in examples:
        grouped['labels'] = [OPTIONS.index(a) for a in examples['answer']]
    return grouped

def to_mc_dataset(df, with_labels=True):
    cols = ['prompt'] + OPTIONS + (['answer'] if with_labels else [])
    ds = Dataset.from_pandas(df[cols])
    ds = ds.map(preprocess_mc, batched=True, remove_columns=ds.column_names)
    ds.set_format('torch')
    return ds

train_mc = to_mc_dataset(train_df)
valid_mc = to_mc_dataset(valid_df)

print("MC datasets ready:", train_mc.num_rows, "train /", valid_mc.num_rows, "valid")
print("input_ids shape per example:", tuple(train_mc[0]['input_ids'].shape), "= (5 options, seq_len)")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1590 [00:00<?, ? examples/s]

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

MC datasets ready: 1590 train / 83 valid
input_ids shape per example: (5, 192) = (5 options, seq_len)


### 10.2 Load DeBERTa-v3-large + LoRA adapters

In [15]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForMultipleChoice.from_pretrained(MC_MODEL)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj",
                    "output.dense", "intermediate.dense"],
    modules_to_save=["classifier", "pooler"],
)

model_mc = get_peft_model(base_model, lora_config)
model_mc.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

trainable params: 8,128,513 || all params: 443,191,298 || trainable%: 1.8341


### 10.3 Training arguments — GPU memory management

DeBERTa-v3-large at seq_len 192 × 5 options is heavy for a 16 GB T4, so:
- `per_device_train_batch_size=2` with `gradient_accumulation_steps=8` → effective batch 16
- `gradient_checkpointing=True` trades compute for memory
- `bf16=True` halves activation memory (and avoids the fp16/LoRA grad-scaler bug)

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mc_lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="wandb",
    run_name="mc-lora-deberta-v3-large",
    bf16=True,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    save_total_limit=1,
)

print("Training args ready. Effective batch:",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

Training args ready. Effective batch: 16


### 10.4 Train

`compute_metrics` here reports **MAP@3 directly** (plus top-1 accuracy) each epoch,
so the model is selected by the actual competition metric.

In [17]:
from transformers import Trainer

def compute_mc_metrics(eval_pred):
    """Top-1 accuracy + MAP@3 computed from the 5-way logits."""
    logits, labels = eval_pred
    order = np.argsort(-logits, axis=1)
    top1 = order[:, 0]

    ap3 = []
    for row_order, true_idx in zip(order, labels):
        rank = np.where(row_order == true_idx)[0][0]
        ap3.append(1.0 / (rank + 1) if rank < 3 else 0.0)

    return {"accuracy": (top1 == labels).mean(), "map3": np.mean(ap3)}

trainer = Trainer(
    model=model_mc,
    args=training_args,
    train_dataset=train_mc,
    eval_dataset=valid_mc,
    processing_class=tokenizer,
    compute_metrics=compute_mc_metrics,
)

trainer.train()

wandb: setting up run 4kp01mfr
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_115037-4kp01mfr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mc-lora-deberta-v3-large
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/4kp01mfr


{'loss': 1.6128, 'grad_norm': 9.77363395690918, 'learning_rate': 5.4545454545454546e-05, 'epoch': 0.5025125628140703}
{'eval_loss': 1.6049091815948486, 'eval_accuracy': 0.39759036144578314, 'eval_map3': 0.5502008032128514, 'eval_runtime': 12.7395, 'eval_samples_per_second': 6.515, 'eval_steps_per_second': 0.863, 'epoch': 0.9849246231155779}
{'train_runtime': 399.4154, 'train_samples_per_second': 3.981, 'train_steps_per_second': 0.123, 'train_loss': 1.6094501261808434, 'epoch': 0.9849246231155779}


TrainOutput(global_step=49, training_loss=1.6094501261808434, metrics={'train_runtime': 399.4154, 'train_samples_per_second': 3.981, 'train_steps_per_second': 0.123, 'train_loss': 1.6094501261808434, 'epoch': 0.9849246231155779})

### 10.5 Score with the notebook's MAP@3 (consistency check + W&B run)

In [18]:
from transformers.integrations import WandbCallback
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks if not isinstance(cb, WandbCallback)
]

def mc_predict(df, trainer, with_labels=True):
    """Rank the 5 options per question using the fine-tuned MC model."""
    ds = to_mc_dataset(df, with_labels=with_labels)
    logits = trainer.predict(ds).predictions
    order = np.argsort(-logits, axis=1)
    return [[OPTIONS[j] for j in row] for row in order]

mc_preds = mc_predict(valid_df, trainer)
mc_score = log_model_run("mc-lora-deberta-v3-large", "fine-tuned", valid_df, mc_preds,
                         config={"base_model": MC_MODEL, "r": 16, "lr": 1e-4,
                                 "epochs": 3, "max_len": MAX_LEN})

Map:   0%|          | 0/83 [00:00<?, ? examples/s]

wandb: Finishing previous runs because reinit is set to True.
wandb: uploading history steps 1-2, summary, console lines 1-2; updating run metadata
wandb: uploading history steps 1-2, summary, console lines 1-2
wandb: uploading summary, console lines 3-3
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁
wandb:               eval/loss ▁
wandb:               eval/map3 ▁
wandb:            eval/runtime ▁
wandb: eval/samples_per_second ▁
wandb:   eval/steps_per_second ▁
wandb:             train/epoch ▁██
wandb:       train/global_step ▁██
wandb:         train/grad_norm ▁
wandb:     train/learning_rate ▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.39759
wandb:               eval/loss 1.60491
wandb:               eval/map3 0.5502
wandb:            eval/runtime 12.7395
wandb: eval/samples_per_second 6.515
wandb:   eval/steps_per_second 0.863
wandb:              total_flos 2813301575884800.0
wandb:             train/epoch 0.98492
wa

[mc-lora-deberta-v3-large] map@3=0.5502 | accuracy=0.3976 | f1_macro=0.3960


## 10.6 Model 5 — Qwen2.5-7B-Instruct *(large LLM, zero-shot)* ⭐

**Why this is the model that can cross 0.73.** The encoders plateaued at ~0.58 because the
bottleneck is *factual scientific knowledge* the small models never learned — and LoRA on a
few thousand examples can't inject knowledge. A 7B instruction-tuned LLM already **contains**
this knowledge from pretraining.

**How we score options:** we show the LLM the question and all 5 options, then read the
**probability it assigns to each letter token (A–E)** as the next token. Ranking those five
probabilities gives our top-3 — no generation parsing needed, and it directly yields a ranking
for MAP@3.

**Memory:** the DeBERTa trainer is deleted first to free GPU memory, then Qwen loads in
**fp16 (~14 GB)** — fits a 16 GB T4/P100. Inference only (no training), so fast and stable.
If loading still OOMs, switch `LLM_NAME` to `Qwen/Qwen2.5-3B-Instruct`.

In [19]:
!pip uninstall -q -y bitsandbytes
print("bitsandbytes removed (not needed - fp16 loading instead).")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


bitsandbytes removed (not needed - fp16 loading instead).


In [20]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer


def _softmax(v):
    v = np.asarray(v, dtype=np.float64)
    e = np.exp(v - v.max())
    return e / e.sum()


def letter_ids(tokenizer):
    space = [tokenizer(f" {L}", add_special_tokens=False)['input_ids'][-1] for L in OPTIONS]
    plain = [tokenizer(f"{L}",  add_special_tokens=False)['input_ids'][-1] for L in OPTIONS]
    return space, plain


def make_orders(n, seed=SEED):
    """Identity ordering plus n-1 distinct random option permutations."""
    rng = np.random.default_rng(seed)
    orders, seen = [list(OPTIONS)], {tuple(OPTIONS)}
    while len(orders) < n:
        p = tuple(rng.permutation(OPTIONS))
        if p not in seen:
            seen.add(p)
            orders.append(list(p))
    return orders[:n]


def build_prompt_ordered(row, order, tokenizer):
    """Format one MCQ, displaying the options in `order` relabelled A-E."""
    opts = "\n".join(f"{OPTIONS[i]}. {row[order[i]]}" for i in range(5))
    user = (f"Answer the multiple-choice question. Reply with only the letter "
            f"of the single best option.\n\nQuestion: {row['prompt']}\n\n{opts}\n\nAnswer:")
    messages = [{"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _letter_probs(last_logits, extraction, ids_space, ids_plain):
    """Length-5 probability vector over the displayed letters A-E."""
    lg = last_logits.float().cpu().numpy()
    if extraction == "space":
        return _softmax(lg[ids_space])
    if extraction == "plain":
        return _softmax(lg[ids_plain])
    if extraction == "both":
        return 0.5 * (_softmax(lg[ids_space]) + _softmax(lg[ids_plain]))
    raise ValueError(f"unknown extraction {extraction!r}")


@torch.no_grad()
def llm_score_matrix(df, model, tokenizer, tta_orders=1, extraction="space", log_every=200):
    """Per-option probabilities as an (n, 5) matrix in canonical A-E order.

    Averages over `tta_orders` option orderings, mapping each displayed-letter
    probability back to the option it actually stood for (position-bias TTA).
    """
    orders = make_orders(tta_orders)
    ids_space, ids_plain = letter_ids(tokenizer)
    out = np.zeros((len(df), 5), dtype=np.float64)
    for n, (_, row) in enumerate(df.iterrows()):
        acc = np.zeros(5)
        for order in orders:
            text = build_prompt_ordered(row, order, tokenizer)
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            last = model(**inputs).logits[0, -1]
            disp = _letter_probs(last, extraction, ids_space, ids_plain)
            for i, orig in enumerate(order):
                acc[OPTIONS.index(orig)] += disp[i]
        out[n] = acc / len(orders)
        if (n + 1) % log_every == 0:
            print(f"  {n+1}/{len(df)} scored")
    return out


def scores_to_preds(score_mat):
    return [[OPTIONS[i] for i in np.argsort(-row)] for row in score_mat]


def free_model(*objs):
    """Drop model references and reclaim GPU memory (T4x2 cannot hold 7B+14B at once)."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()


SCORES = {}  # tag -> {'valid':mat, 'proxy':mat, 'test':mat}

In [21]:
import difflib
from collections import Counter


def build_answer_lookup(train_df, vote="last"):
    """Answer TEXT per core. `last` = baseline dict (last copy wins, reproduces
    0.73607); `majority` votes across copies (differs on only ~13 test rows)."""
    tmp = train_df.assign(
        answer_text=train_df.apply(lambda r: str(r[r['answer']]).strip(), axis=1))
    if vote == "last":
        return dict(zip(tmp['core'], tmp['answer_text']))
    grp = tmp.groupby('core')['answer_text'].apply(list)
    return {core: Counter(v).most_common(1)[0][0] for core, v in grp.items()}


def make_lookup_fn(lut):
    """Build a (row -> (letter, 'exact'|'fuzzy'|None)) retriever over a lookup table.

    Matches by answer TEXT (option order is shuffled train->test). Fuzzy matches
    demand a dominant, high-similarity winner because distractors are minimal edits.
    """
    def look(row):
        if row['core'] not in lut:
            return None, None
        ans = lut[row['core']].strip()
        for o in OPTIONS:
            if str(row[o]).strip() == ans:
                return o, 'exact'
        sims = sorted(((difflib.SequenceMatcher(None, str(row[o]).strip().lower(), ans.lower()).ratio(), o)
                       for o in OPTIONS), reverse=True)
        (s1, o1), (s2, _) = sims[0], sims[1]
        if s1 > 0.9 and (s1 - s2) > 0.05:
            return o1, 'fuzzy'
        return None, None
    return look


answer_lookup = build_answer_lookup(train, CONFIG["LOOKUP_VOTE"])
lookup_with_confidence = make_lookup_fn(answer_lookup)

matches = test.apply(lookup_with_confidence, axis=1)
test['lookup_answer'] = [m[0] for m in matches]
test['lookup_kind'] = [m[1] for m in matches]
print("Test retrieval:", test['lookup_kind'].value_counts(dropna=False).to_dict())


def scores_to_ranking(score_vec):
    return [OPTIONS[i] for i in np.argsort(-np.asarray(score_vec, dtype=float))]


def apply_gating(qwen_scores, lookup_letter, lookup_kind, mode):
    """Combine an LLM probability vector with a retrieval hit into a top-3 list.

    off       : pure LLM ranking.
    always    : exact hit -> rank 1; fuzzy hit -> rank 2 (the 0.73607 policy).
    agreement : exact hit leads only if it is inside the LLM top-2, else the LLM
                top pick leads and the hit drops to rank 2 (guards the ~35% of
                exact hits the noisy train->test transfer gets wrong).
    """
    qwen_rank = scores_to_ranking(qwen_scores)
    if mode == 'off' or lookup_kind is None:
        return qwen_rank[:3]
    if mode == 'always':
        if lookup_kind == 'exact':
            rest = [o for o in qwen_rank if o != lookup_letter]
            return [lookup_letter] + rest[:2]
        top = qwen_rank[0]
        if lookup_letter == top:
            return qwen_rank[:3]
        rest = [o for o in qwen_rank if o not in (top, lookup_letter)]
        return [top, lookup_letter] + rest[:1]
    if mode == 'agreement':
        if lookup_kind == 'exact' and lookup_letter in qwen_rank[:2]:
            rest = [o for o in qwen_rank if o != lookup_letter]
            return [lookup_letter] + rest[:2]
        top = qwen_rank[0]
        if lookup_letter == top:
            return qwen_rank[:3]
        rest = [o for o in qwen_rank if o not in (top, lookup_letter)]
        return [top, lookup_letter] + rest[:1]
    raise ValueError(f"unknown gating mode {mode!r}")


def ensemble_scores(score_mats, weights):
    w = np.asarray(weights, dtype=float); w = w / w.sum()
    return sum(wi * np.asarray(m, dtype=float) for wi, m in zip(w, score_mats))


def build_pseudo_test():
    """Labelled rebuild of the real test's retrieval mechanic (see 12b)."""
    src_rows, qz_rows = [], []
    for _, grp in train.groupby('core'):
        g = grp.reset_index(drop=True)
        if len(g) < 2:
            continue
        src_rows.append(g.iloc[0])
        for k in range(1, len(g)):
            qz_rows.append(g.iloc[k])
    src = pd.DataFrame(src_rows).reset_index(drop=True)
    qz = pd.DataFrame(qz_rows).reset_index(drop=True)
    look = make_lookup_fn(build_answer_lookup(src, CONFIG["LOOKUP_VOTE"]))
    hits = qz.apply(look, axis=1)
    qz['lookup_letter'] = [h[0] for h in hits]
    qz['lookup_kind'] = [h[1] for h in hits]
    return qz

Test retrieval: {'exact': 435, 'fuzzy': 35, None: 30}


In [22]:
def load_llm(name):
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True)
    mdl.eval()
    return mdl, tok


def score_frames(model, tokenizer, frames):
    kw = dict(tta_orders=CONFIG["TTA_ORDERS"], extraction=CONFIG["EXTRACTION"])
    return {tag: llm_score_matrix(df, model, tokenizer, **kw) for tag, df in frames.items()}


try:
    del trainer.model
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

llm, llm_tokenizer = load_llm(CONFIG["LLM_NAME"])
print(f"{CONFIG['LLM_NAME']} loaded in fp16.")

valid_sample = valid_df.head(CONFIG["LLM_VALID_SAMPLE"]).reset_index(drop=True)
frames_7b = {"valid": valid_sample, "test": test}

print("Scoring with 7B ...")
SCORES["7B"] = score_frames(llm, llm_tokenizer, frames_7b)

llm_preds = scores_to_preds(SCORES["7B"]["valid"])
llm_score = log_model_run("qwen2.5-7b-zeroshot", "large-llm-zero-shot", valid_sample, llm_preds,
                          config={"base_model": CONFIG["LLM_NAME"], "dtype": "fp16",
                                  "tta_orders": CONFIG["TTA_ORDERS"],
                                  "extraction": CONFIG["EXTRACTION"]})

pseudo_scores = pseudo_qz = None
if CONFIG["PSEUDO_TEST_EVAL"]:
    pseudo_qz = build_pseudo_test()
    print(f"Scoring pseudo-test ({len(pseudo_qz)} q) with 7B ...")
    pseudo_scores = llm_score_matrix(pseudo_qz, llm, llm_tokenizer,
                                     tta_orders=CONFIG["TTA_ORDERS"], extraction=CONFIG["EXTRACTION"])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen/Qwen2.5-7B-Instruct loaded in fp16.
Scoring with 7B ...
  200/500 scored
  400/500 scored


wandb: setting up run qbkkppsu
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_120221-qbkkppsu
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qwen2.5-7b-zeroshot
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/qbkkppsu
wandb: updating run metadata; uploading summary
wandb: updating run metadata
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.83133
wandb: f1_macro 0.8244
wandb:    map@3 0.90964
wandb: 
wandb: 🚀 View run qwen2.5-7b-zeroshot at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/qbkkppsu
wandb: ⭐️ View project at: https://wandb.ai/23f3004491-indian-institut

[qwen2.5-7b-zeroshot] map@3=0.9096 | accuracy=0.8313 | f1_macro=0.8244


### 10.7 (Optional) Qwen2.5-14B sharded across both T4s

Set `CONFIG["RUN_14B"] = True` to score with 14B in fp16, `device_map="auto"`
(~28 GB across the 2x16 GB T4s). The 7B is **freed first** - 7B+14B (~42 GB) cannot
co-reside on 32 GB, so each model scores every frame while resident and we ensemble
from the saved probability matrices. Guarded so the default path costs nothing.

In [23]:
if CONFIG["RUN_14B"]:
    free_model(llm)
    print(f"GPU free after releasing 7B: "
          f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB")
    llm_14b, llm_tok_14b = load_llm(CONFIG["LLM_NAME_14B"])
    print("Qwen2.5-14B loaded (sharded across both T4s).")
    print("Scoring with 14B ...")
    SCORES["14B"] = score_frames(llm_14b, llm_tok_14b, frames_7b)
    preds_14b = scores_to_preds(SCORES["14B"]["valid"])
    log_model_run("qwen2.5-14b-zeroshot", "large-llm-zero-shot", valid_df, preds_14b,
                  config={"base_model": CONFIG["LLM_NAME_14B"], "dtype": "fp16",
                          "tta_orders": CONFIG["TTA_ORDERS"],
                          "extraction": CONFIG["EXTRACTION"]})
    free_model(llm_14b)
else:
    print("RUN_14B is False - skipping 14B.")

RUN_14B is False - skipping 14B.


In [24]:
if CONFIG["RUN_MODEL2"]:
    free_model(llm)
    gc.collect(); torch.cuda.empty_cache()
    for _d in range(torch.cuda.device_count()):
        _f, _t = torch.cuda.mem_get_info(_d)
        print(f"GPU {_d}: {_f/1e9:.1f} GB free after releasing model 1")

    llm2, llm2_tok = load_llm(CONFIG["MODEL2_NAME"])
    print(f"{CONFIG['MODEL2_NAME']} loaded.")
    print("Scoring with model 2 ...")
    SCORES["M2"] = score_frames(llm2, llm2_tok, frames_7b)

    preds_m2 = scores_to_preds(SCORES["M2"]["valid"])
    log_model_run("mistral-7b-zeroshot", "large-llm-zero-shot", valid_sample, preds_m2,
                  config={"base_model": CONFIG["MODEL2_NAME"], "dtype": "fp16",
                          "tta_orders": CONFIG["TTA_ORDERS"]})
    free_model(llm2)
    gc.collect(); torch.cuda.empty_cache()

GPU 0: 6.2 GB free after releasing model 1
GPU 1: 6.8 GB free after releasing model 1


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

mistralai/Mistral-7B-Instruct-v0.3 loaded.
Scoring with model 2 ...
  200/500 scored
  400/500 scored


wandb: setting up run s23kg4z5
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_121206-s23kg4z5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mistral-7b-zeroshot
wandb: ⭐️ View project at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026
wandb: 🚀 View run at https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/s23kg4z5
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_macro ▁
wandb:    map@3 ▁
wandb: 
wandb: Run summary:
wandb: accuracy 0.79518
wandb: f1_macro 0.78321
wandb:    map@3 0.8755
wandb: 
wandb: 🚀 View run mistral-7b-zeroshot at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f3004491-t22026/runs/s23kg4z5
wandb: ⭐️ View project at: https://wandb.ai/23f3004491-indian-institute-of-technology-madras/23f300

[mistral-7b-zeroshot] map@3=0.8755 | accuracy=0.7952 | f1_macro=0.7832


## 11. Model Comparison

In [25]:
results = pd.DataFrame([
    {"Model": "TF-IDF + cosine",            "Type": "from scratch", "MAP@3": tfidf_score},
    {"Model": "MiniLM bi-encoder",          "Type": "pretrained",   "MAP@3": bi_score},
    {"Model": "NLI cross-encoder",          "Type": "zero-shot",    "MAP@3": ce_score},
    {"Model": "MC LoRA DeBERTa-v3-large",   "Type": "fine-tuned",   "MAP@3": mc_score},
    {"Model": "Qwen2.5-7B-Instruct",        "Type": "large LLM",    "MAP@3": llm_score},
]).sort_values("MAP@3", ascending=False)

print(results.to_string(index=False))
print("\nCutoff to beat: 0.73")

                   Model         Type    MAP@3
     Qwen2.5-7B-Instruct    large LLM 0.909639
MC LoRA DeBERTa-v3-large   fine-tuned 0.550201
       NLI cross-encoder    zero-shot 0.540161
       MiniLM bi-encoder   pretrained 0.443775
         TF-IDF + cosine from scratch 0.357430

Cutoff to beat: 0.73


## 12. Retrieval-Based Duplicate Answering + Final Hybrid Submission

**Data finding (documented in Section 5):** the train file contains each core question in
multiple wrapper phrasings, and once every wrapper variant is stripped correctly, ~98% of
test rows have a matching core in train (~94% resolve to a confident exact/fuzzy retrieval
answer; the rest have a matching core but conflicting/ambiguous answer text across copies).
Answering those via retrieval from the labeled training data is a legitimate
retrieval-augmentation strategy - documented transparently here and in the report.

**Precision safeguards** (first submission with naive fuzzy matching scored 0.734,
suggesting fuzzy matches on minimal-edit distractors are error-prone):
- **Exact answer-text match** -> high precision -> lookup answer ranked first.
- **Fuzzy match** (>0.9 similarity AND dominant by >0.05 over second-best) -> ranked
  second behind Qwen's top pick, cushioning potential errors.
- **Ambiguous or novel** -> Qwen's full ranking.

## 12b. Proxy Evaluation Harness (spend zero submissions)

Two labelled proxies let us rank variants offline:

1. **Novel proxy** (`proxy_df`, train cores absent from test) - measures raw LLM quality
   on questions retrieval cannot help with. This is where 14B / TTA / extraction show up.
2. **Pseudo-test** - rebuilds the real test's retrieval mechanic with labels: copy 0 of
   each repeated core is the lookup source, later copies are held-out questions with their
   own labels.

> **Lesson from the leaderboard:** the pseudo-test favoured `agreement` gating, but on the
> real test `agreement` scored **0.72734 vs 0.73607** for `always`. Train copies are altered
> far less than test options, so the pseudo-test *overstates* how often the lookup is wrong
> and thus overstates agreement's value. **Trust the leaderboard for gating (`always` wins);
> use only the novel-proxy table below for model choice (7B vs 14B vs ensemble).**

In [26]:
def eval_proxy():
    truth = proxy_df['answer'].tolist()
    rows = [("7B", SCORES["7B"]["proxy"])]
    if "14B" in SCORES:
        rows.append(("14B", SCORES["14B"]["proxy"]))
        rows.append(("ens", ensemble_scores(
            [SCORES["7B"]["proxy"], SCORES["14B"]["proxy"]], CONFIG["ENSEMBLE_WEIGHTS"])))
    print("=== Novel-proxy MAP@3 (no lookup; pure model quality) ===")
    for name, mat in rows:
        print(f"  {name:>4}: {mean_average_precision_at_3(truth, scores_to_preds(mat)):.4f}")


def eval_pseudo_gating():
    truth = pseudo_qz['answer'].tolist()
    exact = pseudo_qz['lookup_kind'] == 'exact'
    acc = np.mean([pseudo_qz.iloc[i]['lookup_letter'] == truth[i]
                   for i in range(len(pseudo_qz)) if exact.iloc[i]])
    print(f"=== Pseudo-test ({len(pseudo_qz)} q, exact-lookup transfer-acc={acc:.3f}) ===")
    print("    (optimistic vs real test: train copies are altered less than test options)")
    for mode in ['off', 'always', 'agreement']:
        preds = [apply_gating(pseudo_scores[i], pseudo_qz.iloc[i]['lookup_letter'],
                              pseudo_qz.iloc[i]['lookup_kind'], mode)
                 for i in range(len(pseudo_qz))]
        print(f"  gating={mode:<10} MAP@3={mean_average_precision_at_3(truth, preds):.4f}")


def report_test_agreement(test_scores):
    """Unlabelled sanity check: how often exact lookup sits in the LLM top-1/2 on real test."""
    idx = np.where((test['lookup_kind'] == 'exact').values)[0]
    if not len(idx):
        return
    top1 = sum(test.iloc[i]['lookup_answer'] == scores_to_ranking(test_scores[i])[0] for i in idx)
    top2 = sum(test.iloc[i]['lookup_answer'] in scores_to_ranking(test_scores[i])[:2] for i in idx)
    print(f"=== Test exact-lookup vs LLM (n={len(idx)}) ===")
    print(f"  lookup in LLM top-1: {top1/len(idx):.3f} | top-2: {top2/len(idx):.3f}")
    print("  (low top-1 => many exact hits disagree with the LLM => agreement gating matters)")


if CONFIG["PROXY_EVAL"] and "proxy" in SCORES["7B"]:
    eval_proxy()
if CONFIG["PSEUDO_TEST_EVAL"] and pseudo_scores is not None:
    eval_pseudo_gating()

if CONFIG.get("RUN_MODEL2") and "M2" in SCORES:
    v_q = mean_average_precision_at_3(valid_sample['answer'].tolist(), scores_to_preds(SCORES["7B"]["valid"]))
    v_m = mean_average_precision_at_3(valid_sample['answer'].tolist(), scores_to_preds(SCORES["M2"]["valid"]))
    ens_v = ensemble_scores([SCORES["7B"]["valid"], SCORES["M2"]["valid"]], CONFIG["ENSEMBLE_W_QWEN_M2"])
    v_e = mean_average_precision_at_3(valid_sample['answer'].tolist(), scores_to_preds(ens_v))
    print(f"Valid MAP@3  Qwen={v_q:.4f}  Mistral={v_m:.4f}  Ensemble={v_e:.4f}")

Valid MAP@3  Qwen=0.9096  Mistral=0.8755  Ensemble=0.9157


## 13. Final Submission (config-driven)

Uses `CONFIG["GATING"]` to combine retrieval with the LLM, and optionally the
7B+14B ensemble. The unlabelled top-1/top-2 agreement report above is printed for
the exact test scores actually used here.

In [27]:
wandb.finish()

if CONFIG.get("RUN_MODEL2") and "M2" in SCORES:
    test_scores = ensemble_scores([SCORES["7B"]["test"], SCORES["M2"]["test"]],
                                  CONFIG["ENSEMBLE_W_QWEN_M2"])
    used = "Qwen7B + Mistral7B ensemble"
elif CONFIG["ENSEMBLE_7B_14B"] and "14B" in SCORES:
    test_scores = ensemble_scores([SCORES["7B"]["test"], SCORES["14B"]["test"]],
                                  CONFIG["ENSEMBLE_WEIGHTS"])
    used = "7B+14B ensemble"
elif "14B" in SCORES:
    test_scores = SCORES["14B"]["test"]; used = "14B"
else:
    test_scores = SCORES["7B"]["test"]; used = "7B"

report_test_agreement(test_scores)

final_preds = [apply_gating(test_scores[i], test.iloc[i]['lookup_answer'],
                            test.iloc[i]['lookup_kind'], CONFIG["GATING"])
               for i in range(len(test))]

submission = pd.DataFrame({"ID": test['id'],
                           "Prediction": [" ".join(p[:3]) for p in final_preds]})
submission.to_csv("submission.csv", index=False)

assert list(submission.columns) == ["ID", "Prediction"]
assert submission['Prediction'].str.split().str.len().eq(3).all()
assert len(submission) == len(test)
assert submission['Prediction'].apply(lambda s: all(t in OPTIONS for t in s.split())).all()
print(submission.head())
print(f"\nRows: {len(submission)} | model={used} | gating={CONFIG['GATING']} - submission.csv ready.")

=== Test exact-lookup vs LLM (n=435) ===
  lookup in LLM top-1: 0.793 | top-2: 0.926
  (low top-1 => many exact hits disagree with the LLM => agreement gating matters)
   ID Prediction
0   1      A E D
1   2      B A E
2   3      B C D
3   4      E A C
4   5      C A B

Rows: 500 | model=Qwen7B + Mistral7B ensemble | gating=always - submission.csv ready.


## 14. (Optional) Ensemble — Milestone 5

If two models are individually strong, averaging their per-option scores can beat either alone.
Here we average normalized ranks from the LLM and the cross-encoder. Only worth submitting if
it beats the best single model on validation.

In [28]:
def rank_scores(predictions):
    """Convert each ranked letter-list into a per-option score (higher = better)."""
    score_rows = []
    for pred in predictions:
        s = {L: 0 for L in OPTIONS}
        for rank, L in enumerate(pred):
            s[L] = len(OPTIONS) - rank
        score_rows.append([s[L] for L in OPTIONS])
    return np.array(score_rows, dtype=float)


def ensemble(pred_lists, weights):
    """Weighted-average several models' rank-scores, then re-rank."""
    combined = sum(w * rank_scores(p) for w, p in zip(weights, pred_lists))
    return [[OPTIONS[i] for i in np.argsort(row)[::-1]] for row in combined]


truth = valid_sample['answer'].tolist()
if "14B" in SCORES:
    ens_val = scores_to_preds(ensemble_scores(
        [SCORES["7B"]["valid"], SCORES["14B"]["valid"]], CONFIG["ENSEMBLE_WEIGHTS"]))
    ens_score = mean_average_precision_at_3(truth, ens_val)
    map14 = mean_average_precision_at_3(truth, scores_to_preds(SCORES["14B"]["valid"]))
    print(f"Ensemble (7B+14B probs) MAP@3: {ens_score:.4f}")
    print(f"Best single (7B={llm_score:.4f}, 14B={map14:.4f})")
    print("Use ensemble?", "YES" if ens_score > max(llm_score, map14) else "no - single model better")
else:
    print(f"Best single LLM (7B) MAP@3 on valid sample: {llm_score:.4f}")
    print("Primary submission model is the Qwen7B + Mistral7B ensemble (see Valid MAP@3 above).")

Best single LLM (7B) MAP@3 on valid sample: 0.9096
Primary submission model is the Qwen7B + Mistral7B ensemble (see Valid MAP@3 above).
